In [1]:
# Cell 1 — Locate the project and confirm the Python environment.
#
# This cell does not read or modify the large tick-data file.
# It only establishes the paths we will use later.

from pathlib import Path
import sys

import numpy as np
import pandas as pd


# Jupyter may start in either the repository root or its notebooks folder.
# Search upward until we find both .git and raw_data.
current_directory = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in [current_directory, *current_directory.parents]
        if (directory / ".git").exists()
        and (directory / "raw_data").exists()
    ),
    None,
)


# Stop immediately if Jupyter was launched outside the project.
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Start JupyterLab from the EURUSD 15 folder."
    )


# Define the main project paths.
RAW_DATA_DIR = PROJECT_ROOT / "raw_data"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"


# Display enough information to verify the setup.
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Project root:", PROJECT_ROOT)
print("Raw data directory exists:", RAW_DATA_DIR.exists())

Python version: 3.13.16
Pandas version: 3.0.6
NumPy version: 2.5.3
Project root: C:\Users\Ijzd4\trend models\EURUSD 15
Raw data directory exists: True


In [2]:
# Cell 2 — Inventory the CSV files in raw_data.
#
# This cell reads only file metadata:
# - filename
# - size
# - last modification time
#
# It does not open or load any CSV data.

from datetime import datetime


# Find CSV files located directly inside raw_data.
# Diagnostic subfolders and cache files are intentionally excluded.
csv_files = sorted(RAW_DATA_DIR.glob("*.csv*"))


# Build a compact inventory.
file_inventory = []

for file_path in csv_files:
    file_size_bytes = file_path.stat().st_size

    file_inventory.append(
        {
            "filename": file_path.name,
            "size_gb": file_size_bytes / (1024 ** 3),
            "size_bytes": file_size_bytes,
            "last_modified": datetime.fromtimestamp(
                file_path.stat().st_mtime
            ),
        }
    )


# Convert the inventory to a table for easier inspection.
inventory_df = pd.DataFrame(file_inventory)


if inventory_df.empty:
    print("No CSV files were found in:", RAW_DATA_DIR)
else:
    display(
        inventory_df[
            ["filename", "size_gb", "size_bytes", "last_modified"]
        ].style.format(
            {
                "size_gb": "{:.3f}",
                "size_bytes": "{:,}",
            }
        )
    )

,filename,size_gb,size_bytes,last_modified
0,eurusd-test-bi5-2016-04-05.csv,0.000,0,2026-10-05 20:34:49.653091
1,eurusd-tick-2010-01-01-2026-10-01.csv,4.389,"4,712,645,339",2026-10-04 21:33:17.431026
2,eurusd-tick-2016-04-05-2017-01-01.csv,0.001,"606,292",2026-10-05 20:28:59.469549
3,eurusd-tick-2016-04-05-2026-10-01.csv.csv,0.000,0,2026-10-05 20:23:29.315378
4,eurusd-tick-2026-09-01-2026-09-02.csv,0.002,"1,741,364",2026-10-04 19:42:10.954492


In [3]:
# Cell 3 — Inspect the boundaries of the main CSV efficiently.
#
# We only read:
# - the header
# - the first data row
# - the final data row
#
# Seeking directly to the end avoids loading or scanning the 4.4 GB file.

MAIN_RAW_FILE = (
    RAW_DATA_DIR
    / "eurusd-tick-2010-01-01-2026-10-01.csv"
)


# Confirm that we selected an existing, non-empty file.
if not MAIN_RAW_FILE.exists():
    raise FileNotFoundError(MAIN_RAW_FILE)

if MAIN_RAW_FILE.stat().st_size == 0:
    raise ValueError("The selected file is empty.")


with MAIN_RAW_FILE.open("rb") as csv_file:
    # Read the header and first data record.
    header = csv_file.readline().decode("utf-8").strip()
    first_row = csv_file.readline().decode("utf-8").strip()

    # Read a small block from the end of the file.
    # A normal tick row is much shorter than 8 KB.
    bytes_to_read = min(MAIN_RAW_FILE.stat().st_size, 8192)

    csv_file.seek(-bytes_to_read, 2)
    final_block = csv_file.read().decode("utf-8")

    # Ignore blank lines and retain the final complete row.
    final_lines = [
        line.strip()
        for line in final_block.splitlines()
        if line.strip()
    ]

    last_row = final_lines[-1]


print("Header:")
print(header)

print("\nFirst data row:")
print(first_row)

print("\nLast data row:")
print(last_row)

Header:
timestamp,askPrice,bidPrice,askVolume,bidVolume

First data row:
1262304003964,1.43293,1.43283,3,2.3

Last data row:
1459835999593,1.13851,1.1384,1,2.5


In [4]:
# Cell 4 — Calculate the actual time coverage of the main file.
#
# Dukascopy timestamps are Unix timestamps measured in milliseconds.
# We convert them explicitly to timezone-aware UTC timestamps.

EXPECTED_COLUMN_COUNT = 5


# Split each row into its five CSV fields.
first_values = first_row.split(",")
last_values = last_row.split(",")


# Validate the boundary rows before using them.
if len(first_values) != EXPECTED_COLUMN_COUNT:
    raise ValueError(
        f"First row has {len(first_values)} columns; "
        f"expected {EXPECTED_COLUMN_COUNT}."
    )

if len(last_values) != EXPECTED_COLUMN_COUNT:
    raise ValueError(
        f"Last row has {len(last_values)} columns; "
        f"expected {EXPECTED_COLUMN_COUNT}."
    )


# Convert the first field of each row from milliseconds to UTC.
first_timestamp = pd.to_datetime(
    int(first_values[0]),
    unit="ms",
    utc=True,
)

last_timestamp = pd.to_datetime(
    int(last_values[0]),
    unit="ms",
    utc=True,
)


# Calculate the observed duration.
observed_duration = last_timestamp - first_timestamp
approximate_years = observed_duration.days / 365.25


print("First tick:", first_timestamp)
print("Last tick: ", last_timestamp)
print("Observed days:", observed_duration.days)
print(f"Approximate years: {approximate_years:.2f}")

First tick: 2010-01-01 00:00:03.964000+00:00
Last tick:  2016-04-05 05:59:59.593000+00:00
Observed days: 2286
Approximate years: 6.26


In [5]:
# Cell 5 — Inspect the time boundaries of every non-empty CSV.
#
# This function reads only the first and last records of each file.
# It remains fast even when a file is several gigabytes.

def inspect_csv_boundaries(file_path):
    """Return basic boundary information without loading the full CSV."""

    file_size = file_path.stat().st_size

    # Empty files have no header or observations to inspect.
    if file_size == 0:
        return {
            "filename": file_path.name,
            "size_mb": 0,
            "first_tick": pd.NaT,
            "last_tick": pd.NaT,
            "status": "empty",
        }

    with file_path.open("rb") as csv_file:
        header_line = csv_file.readline().decode("utf-8").strip()
        first_line = csv_file.readline().decode("utf-8").strip()

        bytes_to_read = min(file_size, 8192)
        csv_file.seek(-bytes_to_read, 2)

        ending_lines = [
            line.strip()
            for line in csv_file.read().decode("utf-8").splitlines()
            if line.strip()
        ]

        last_line = ending_lines[-1]

    # Convert the boundary timestamps to UTC.
    first_tick = pd.to_datetime(
        int(first_line.split(",")[0]),
        unit="ms",
        utc=True,
    )

    last_tick = pd.to_datetime(
        int(last_line.split(",")[0]),
        unit="ms",
        utc=True,
    )

    return {
        "filename": file_path.name,
        "size_mb": file_size / (1024 ** 2),
        "first_tick": first_tick,
        "last_tick": last_tick,
        "status": "non-empty",
    }


coverage_records = [
    inspect_csv_boundaries(file_path)
    for file_path in csv_files
]

coverage_df = pd.DataFrame(coverage_records)


display(
    coverage_df.style.format(
        {
            "size_mb": "{:,.2f}",
        }
    )
)

,filename,size_mb,first_tick,last_tick,status
0,eurusd-test-bi5-2016-04-05.csv,0.00,NaT,NaT,empty
1,eurusd-tick-2010-01-01-2026-10-01.csv,"4,494.33",2010-01-01 00:00:03.964000+00:00,2016-04-05 05:59:59.593000+00:00,non-empty
2,eurusd-tick-2016-04-05-2017-01-01.csv,0.58,2016-04-05 00:00:01.589000+00:00,2016-04-05 05:59:59.593000+00:00,non-empty
3,eurusd-tick-2016-04-05-2026-10-01.csv.csv,0.00,NaT,NaT,empty
4,eurusd-tick-2026-09-01-2026-09-02.csv,1.66,2026-09-01 00:00:00.118000+00:00,2026-09-01 23:59:59.083000+00:00,non-empty


In [6]:
# Cell 6 — Load a small sample from the main tick file.
#
# We explicitly define each data type instead of asking pandas
# to guess types from a multi-gigabyte file.

TICK_DTYPES = {
    "timestamp": "int64",
    "askPrice": "float64",
    "bidPrice": "float64",
    "askVolume": "float64",
    "bidVolume": "float64",
}

SAMPLE_ROW_COUNT = 100_000


tick_sample = pd.read_csv(
    MAIN_RAW_FILE,
    nrows=SAMPLE_ROW_COUNT,
    dtype=TICK_DTYPES,
)


print("Requested rows:", f"{SAMPLE_ROW_COUNT:,}")
print("Loaded rows:   ", f"{len(tick_sample):,}")
print("\nColumn types:")
print(tick_sample.dtypes)

print("\nApproximate sample memory:")
print(f"{tick_sample.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


display(tick_sample.head())

Requested rows: 100,000
Loaded rows:    100,000

Column types:
timestamp      int64
askPrice     float64
bidPrice     float64
askVolume    float64
bidVolume    float64
dtype: object

Approximate sample memory:
3.81 MB


,timestamp,askPrice,bidPrice,askVolume,bidVolume
0,1262304003964,1.43293,1.43283,3.0,2.3
1,1262304005996,1.43290,1.43278,4.2,1.4
2,1262304010385,1.43287,1.43274,4.2,1.9
3,1262304015255,1.43285,1.43273,3.4,1.5
4,1262304019201,1.43285,1.43273,4.4,1.7


In [7]:
# Cell 7 — Run quality checks on the 100,000-row sample.
#
# Duplicate timestamps are not automatically errors:
# multiple legitimate quote updates can share the same millisecond.
# Exact duplicate rows are a more meaningful duplication check.

sample_timestamp_diff = tick_sample["timestamp"].diff()

sample_spread = (
    tick_sample["askPrice"]
    - tick_sample["bidPrice"]
)

# EUR/USD has a conventional pip size of 0.0001.
sample_spread_pips = sample_spread / 0.0001


print("Sample coverage:")
print(
    pd.to_datetime(
        tick_sample["timestamp"].iloc[0],
        unit="ms",
        utc=True,
    )
)
print(
    pd.to_datetime(
        tick_sample["timestamp"].iloc[-1],
        unit="ms",
        utc=True,
    )
)

print("\nQuality checks:")
print("Missing values:", int(tick_sample.isna().sum().sum()))
print("Exact duplicate rows:", int(tick_sample.duplicated().sum()))
print(
    "Duplicate timestamps:",
    int(tick_sample["timestamp"].duplicated().sum()),
)
print(
    "Out-of-order timestamps:",
    int((sample_timestamp_diff < 0).sum()),
)
print("Ask below bid:", int((sample_spread < 0).sum()))
print("Ask equal to bid:", int((sample_spread == 0).sum()))
print(
    "Non-positive prices:",
    int(
        (
            (tick_sample["askPrice"] <= 0)
            | (tick_sample["bidPrice"] <= 0)
        ).sum()
    ),
)
print(
    "Negative volumes:",
    int(
        (
            (tick_sample["askVolume"] < 0)
            | (tick_sample["bidVolume"] < 0)
        ).sum()
    ),
)

print("\nSpread distribution in pips:")
display(
    sample_spread_pips.describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).to_frame(name="spread_pips")
)

Sample coverage:
2010-01-01 00:00:03.964000+00:00
2010-01-06 15:27:18.033000+00:00

Quality checks:
Missing values: 0
Exact duplicate rows: 0
Duplicate timestamps: 0
Out-of-order timestamps: 0
Ask below bid: 0
Ask equal to bid: 0
Non-positive prices: 0
Negative volumes: 0

Spread distribution in pips:


,spread_pips
count,100000.000000
mean,0.930797
std,0.306180
min,0.100000
1%,0.400000
5%,0.400000
50%,1.000000
95%,1.400000
99%,1.800000
max,5.200000


In [8]:
# Cell 8 — Stream through the entire CSV and calculate quality totals.
#
# This is a read-only operation.
# The complete 4.4 GB file is never held in memory at once.
#
# Jupyter will show [*] while this cell is running.
# A progress message is printed after every 10 million rows.

CHUNK_SIZE = 1_000_000

total_rows = 0
total_missing_values = 0
total_adjacent_duplicate_rows = 0
total_duplicate_timestamps = 0
total_out_of_order_timestamps = 0
total_ask_below_bid = 0
total_ask_equal_bid = 0
total_non_positive_prices = 0
total_negative_volumes = 0

minimum_spread_pips = np.inf
maximum_spread_pips = -np.inf

first_timestamp_ms = None
last_timestamp_ms = None
previous_final_row = None


chunk_reader = pd.read_csv(
    MAIN_RAW_FILE,
    dtype=TICK_DTYPES,
    chunksize=CHUNK_SIZE,
)


for chunk_number, chunk in enumerate(chunk_reader, start=1):
    row_count = len(chunk)
    total_rows += row_count

    timestamps = chunk["timestamp"].to_numpy()
    ask_prices = chunk["askPrice"].to_numpy()
    bid_prices = chunk["bidPrice"].to_numpy()
    ask_volumes = chunk["askVolume"].to_numpy()
    bid_volumes = chunk["bidVolume"].to_numpy()

    if first_timestamp_ms is None:
        first_timestamp_ms = int(timestamps[0])

    last_timestamp_ms = int(timestamps[-1])

    # Count missing fields.
    total_missing_values += int(
        chunk.isna().to_numpy().sum()
    )

    # Check timestamp order and repeated timestamps inside the chunk.
    timestamp_differences = np.diff(timestamps)

    total_duplicate_timestamps += int(
        np.count_nonzero(timestamp_differences == 0)
    )

    total_out_of_order_timestamps += int(
        np.count_nonzero(timestamp_differences < 0)
    )

    # Compare complete neighboring rows inside the chunk.
    chunk_values = chunk.to_numpy()

    total_adjacent_duplicate_rows += int(
        np.count_nonzero(
            np.all(
                chunk_values[1:] == chunk_values[:-1],
                axis=1,
            )
        )
    )

    # Check the boundary between this chunk and the previous chunk.
    if previous_final_row is not None:
        first_current_row = chunk_values[0]

        if timestamps[0] == previous_final_row[0]:
            total_duplicate_timestamps += 1

        if timestamps[0] < previous_final_row[0]:
            total_out_of_order_timestamps += 1

        if np.array_equal(first_current_row, previous_final_row):
            total_adjacent_duplicate_rows += 1

    previous_final_row = chunk_values[-1].copy()

    # Validate prices, spreads, and volumes.
    spreads = ask_prices - bid_prices
    spread_pips = spreads / 0.0001

    total_ask_below_bid += int(
        np.count_nonzero(spreads < 0)
    )

    total_ask_equal_bid += int(
        np.count_nonzero(spreads == 0)
    )

    total_non_positive_prices += int(
        np.count_nonzero(
            (ask_prices <= 0) | (bid_prices <= 0)
        )
    )

    total_negative_volumes += int(
        np.count_nonzero(
            (ask_volumes < 0) | (bid_volumes < 0)
        )
    )

    minimum_spread_pips = min(
        minimum_spread_pips,
        float(np.nanmin(spread_pips)),
    )

    maximum_spread_pips = max(
        maximum_spread_pips,
        float(np.nanmax(spread_pips)),
    )

    # Print progress every ten chunks.
    if chunk_number % 10 == 0:
        print(f"Processed {total_rows:,} rows...")


first_full_timestamp = pd.to_datetime(
    first_timestamp_ms,
    unit="ms",
    utc=True,
)

last_full_timestamp = pd.to_datetime(
    last_timestamp_ms,
    unit="ms",
    utc=True,
)


print("\nFull-file audit complete")
print("------------------------")
print("Total rows:", f"{total_rows:,}")
print("First tick:", first_full_timestamp)
print("Last tick:", last_full_timestamp)
print("Missing values:", f"{total_missing_values:,}")
print(
    "Adjacent exact duplicate rows:",
    f"{total_adjacent_duplicate_rows:,}",
)
print(
    "Duplicate timestamps:",
    f"{total_duplicate_timestamps:,}",
)
print(
    "Out-of-order timestamps:",
    f"{total_out_of_order_timestamps:,}",
)
print("Ask below bid:", f"{total_ask_below_bid:,}")
print("Ask equal to bid:", f"{total_ask_equal_bid:,}")
print(
    "Non-positive prices:",
    f"{total_non_positive_prices:,}",
)
print(
    "Negative volumes:",
    f"{total_negative_volumes:,}",
)
print(f"Minimum spread: {minimum_spread_pips:.2f} pips")
print(f"Maximum spread: {maximum_spread_pips:.2f} pips")

Processed 10,000,000 rows...
Processed 20,000,000 rows...
Processed 30,000,000 rows...
Processed 40,000,000 rows...
Processed 50,000,000 rows...
Processed 60,000,000 rows...
Processed 70,000,000 rows...
Processed 80,000,000 rows...
Processed 90,000,000 rows...
Processed 100,000,000 rows...
Processed 110,000,000 rows...
Processed 120,000,000 rows...

Full-file audit complete
------------------------
Total rows: 123,790,754
First tick: 2010-01-01 00:00:03.964000+00:00
Last tick: 2016-04-05 05:59:59.593000+00:00
Missing values: 0
Adjacent exact duplicate rows: 0
Duplicate timestamps: 0
Out-of-order timestamps: 0
Ask below bid: 0
Ask equal to bid: 770
Non-positive prices: 0
Negative volumes: 0
Minimum spread: 0.00 pips
Maximum spread: 34.30 pips


In [9]:
# Cell 9 — Summarize full-file data-quality rates.
#
# This cell uses the counters already calculated by Cell 8.
# It does not reread the raw CSV.

quality_summary = pd.DataFrame(
    {
        "check": [
            "Missing values",
            "Adjacent exact duplicate rows",
            "Duplicate timestamps",
            "Out-of-order timestamps",
            "Ask below bid",
            "Ask equal to bid",
            "Non-positive prices",
            "Negative volumes",
        ],
        "count": [
            total_missing_values,
            total_adjacent_duplicate_rows,
            total_duplicate_timestamps,
            total_out_of_order_timestamps,
            total_ask_below_bid,
            total_ask_equal_bid,
            total_non_positive_prices,
            total_negative_volumes,
        ],
    }
)


# Express each issue count as a percentage of all tick rows.
quality_summary["percentage_of_rows"] = (
    quality_summary["count"]
    / total_rows
    * 100
)


display(
    quality_summary.style.format(
        {
            "count": "{:,}",
            "percentage_of_rows": "{:.6f}%",
        }
    )
)

print("Audit conclusion:")
print("The raw tick file is suitable for minute-bar construction.")

,check,count,percentage_of_rows
0,Missing values,0,0.000000%
1,Adjacent exact duplicate rows,0,0.000000%
2,Duplicate timestamps,0,0.000000%
3,Out-of-order timestamps,0,0.000000%
4,Ask below bid,0,0.000000%
5,Ask equal to bid,770,0.000622%
6,Non-positive prices,0,0.000000%
7,Negative volumes,0,0.000000%


Audit conclusion:
The raw tick file is suitable for minute-bar construction.


In [11]:
# Cell 10 — Derive tick-level fields using exact price-point arithmetic.
#
# EUR/USD Dukascopy quotes use five decimal places.
# Converting prices to integer points prevents floating-point noise
# from creating false upticks or downticks.

sample_ticks = tick_sample.copy()

PRICE_SCALE = 100_000
POINTS_PER_PIP = 10


# Convert Unix milliseconds to timezone-aware UTC timestamps.
sample_ticks["event_time"] = pd.to_datetime(
    sample_ticks["timestamp"],
    unit="ms",
    utc=True,
)

sample_ticks["minute_start"] = (
    sample_ticks["event_time"].dt.floor("min")
)


# Convert bid and ask prices to exact integer points.
#
# Example:
# 1.43293 becomes 143293 points.
sample_ticks["ask_points"] = np.rint(
    sample_ticks["askPrice"] * PRICE_SCALE
).astype("int64")

sample_ticks["bid_points"] = np.rint(
    sample_ticks["bidPrice"] * PRICE_SCALE
).astype("int64")


# The sum of ask_points and bid_points represents twice the midpoint.
# Keeping this as an integer allows exact directional comparisons.
sample_ticks["mid_half_points"] = (
    sample_ticks["ask_points"]
    + sample_ticks["bid_points"]
)


# Convert back to a conventional decimal midpoint for reporting.
sample_ticks["mid"] = (
    sample_ticks["mid_half_points"]
    / (2 * PRICE_SCALE)
)


# One EUR/USD pip equals ten five-decimal price points.
sample_ticks["spread_pips"] = (
    sample_ticks["ask_points"]
    - sample_ticks["bid_points"]
) / POINTS_PER_PIP


# Calculate exact midpoint direction from integer differences.
midpoint_point_changes = np.diff(
    sample_ticks["mid_half_points"].to_numpy(),
    prepend=sample_ticks["mid_half_points"].iloc[0],
)

sample_ticks["tick_direction"] = np.sign(
    midpoint_point_changes
).astype("int8")


sample_ticks["is_uptick"] = (
    sample_ticks["tick_direction"] == 1
)

sample_ticks["is_downtick"] = (
    sample_ticks["tick_direction"] == -1
)

sample_ticks["is_unchanged"] = (
    sample_ticks["tick_direction"] == 0
)

sample_ticks["is_locked_quote"] = (
    sample_ticks["ask_points"]
    == sample_ticks["bid_points"]
)


display(
    sample_ticks[
        [
            "event_time",
            "minute_start",
            "askPrice",
            "bidPrice",
            "mid",
            "spread_pips",
            "tick_direction",
        ]
    ].head(10)
)

,event_time,minute_start,askPrice,bidPrice,mid,spread_pips,tick_direction
0,2010-01-01 00:00:03.964000+00:00,2010-01-01 00:00:00+00:00,1.43293,1.43283,1.432880,1.0,0
1,2010-01-01 00:00:05.996000+00:00,2010-01-01 00:00:00+00:00,1.43290,1.43278,1.432840,1.2,-1
2,2010-01-01 00:00:10.385000+00:00,2010-01-01 00:00:00+00:00,1.43287,1.43274,1.432805,1.3,-1
3,2010-01-01 00:00:15.255000+00:00,2010-01-01 00:00:00+00:00,1.43285,1.43273,1.432790,1.2,-1
4,2010-01-01 00:00:19.201000+00:00,2010-01-01 00:00:00+00:00,1.43285,1.43273,1.432790,1.2,0
5,2010-01-01 00:00:19.544000+00:00,2010-01-01 00:00:00+00:00,1.43283,1.43273,1.432780,1.0,-1
6,2010-01-01 00:00:29.027000+00:00,2010-01-01 00:00:00+00:00,1.43285,1.43273,1.432790,1.2,1
7,2010-01-01 00:00:34.424000+00:00,2010-01-01 00:00:00+00:00,1.43284,1.43274,1.432790,1.0,0
8,2010-01-01 00:00:37.325000+00:00,2010-01-01 00:00:00+00:00,1.43284,1.43274,1.432790,1.0,0
9,2010-01-01 00:00:38.758000+00:00,2010-01-01 00:00:00+00:00,1.43284,1.43272,1.432780,1.2,-1


In [12]:
# Cell 11 — Aggregate the sample ticks into one-minute observations.
#
# Each row represents the half-open interval:
# [minute_start, minute_start + 1 minute)
#
# The information becomes usable only at available_at.

minute_sample = (
    sample_ticks
    .groupby("minute_start", sort=True)
    .agg(
        # Midpoint OHLC
        mid_open=("mid", "first"),
        mid_high=("mid", "max"),
        mid_low=("mid", "min"),
        mid_close=("mid", "last"),

        # Bid OHLC
        bid_open=("bidPrice", "first"),
        bid_high=("bidPrice", "max"),
        bid_low=("bidPrice", "min"),
        bid_close=("bidPrice", "last"),

        # Ask OHLC
        ask_open=("askPrice", "first"),
        ask_high=("askPrice", "max"),
        ask_low=("askPrice", "min"),
        ask_close=("askPrice", "last"),

        # Spread behavior during the minute
        spread_open_pips=("spread_pips", "first"),
        spread_mean_pips=("spread_pips", "mean"),
        spread_median_pips=("spread_pips", "median"),
        spread_max_pips=("spread_pips", "max"),
        spread_close_pips=("spread_pips", "last"),

        # Quote activity and direction
        tick_count=("timestamp", "size"),
        uptick_count=("is_uptick", "sum"),
        downtick_count=("is_downtick", "sum"),
        unchanged_count=("is_unchanged", "sum"),
        locked_quote_count=("is_locked_quote", "sum"),

        # Top-of-book volume summaries
        ask_volume_mean=("askVolume", "mean"),
        bid_volume_mean=("bidVolume", "mean"),
        ask_volume_close=("askVolume", "last"),
        bid_volume_close=("bidVolume", "last"),
    )
    .reset_index()
)


# The bar is not complete until the following minute begins.
minute_sample["available_at"] = (
    minute_sample["minute_start"]
    + pd.Timedelta(minutes=1)
)


# Tick imbalance excludes unchanged midpoint updates.
directional_tick_count = (
    minute_sample["uptick_count"]
    + minute_sample["downtick_count"]
)

minute_sample["tick_imbalance"] = np.where(
    directional_tick_count > 0,
    (
        minute_sample["uptick_count"]
        - minute_sample["downtick_count"]
    ) / directional_tick_count,
    0.0,
)


# Express quote intensity on a per-second basis.
minute_sample["ticks_per_second"] = (
    minute_sample["tick_count"] / 60
)


print("Sample ticks:", f"{len(sample_ticks):,}")
print("One-minute observations:", f"{len(minute_sample):,}")
print("First minute:", minute_sample["minute_start"].iloc[0])
print("Last minute: ", minute_sample["minute_start"].iloc[-1])


display(
    minute_sample[
        [
            "minute_start",
            "available_at",
            "mid_open",
            "mid_high",
            "mid_low",
            "mid_close",
            "spread_median_pips",
            "spread_max_pips",
            "tick_count",
            "tick_imbalance",
        ]
    ].head()
)

Sample ticks: 100,000
One-minute observations: 5,008
First minute: 2010-01-01 00:00:00+00:00
Last minute:  2010-01-06 15:27:00+00:00


,minute_start,available_at,mid_open,mid_high,mid_low,mid_close,spread_median_pips,spread_max_pips,tick_count,tick_imbalance
0,2010-01-01 00:00:00+00:00,2010-01-01 00:01:00+00:00,1.43288,1.43288,1.43272,1.43272,1.05,1.3,14,-0.800000
1,2010-01-01 00:01:00+00:00,2010-01-01 00:02:00+00:00,1.43273,1.43290,1.43273,1.43290,1.00,1.3,13,1.000000
2,2010-01-01 00:02:00+00:00,2010-01-01 00:03:00+00:00,1.43287,1.43287,1.43274,1.43287,1.20,1.4,10,0.111111
3,2010-01-01 00:03:00+00:00,2010-01-01 00:04:00+00:00,1.43287,1.43287,1.43274,1.43279,1.10,1.4,15,-0.166667
4,2010-01-01 00:04:00+00:00,2010-01-01 00:05:00+00:00,1.43288,1.43288,1.43282,1.43282,1.00,1.2,17,-0.285714


In [13]:
# Cell 12 — Reconcile the sample ticks with the minute observations.
#
# Every check should return PASS.
# This cell does not reread the raw CSV.

validation_checks = {
    # Every source tick must belong to exactly one minute.
    "Tick counts reconcile": (
        minute_sample["tick_count"].sum()
        == len(sample_ticks)
    ),

    # Every tick must be classified as up, down, or unchanged.
    "Direction counts reconcile": (
        (
            minute_sample["uptick_count"]
            + minute_sample["downtick_count"]
            + minute_sample["unchanged_count"]
        ).sum()
        == len(sample_ticks)
    ),

    # There must be only one output row per observed minute.
    "Minute timestamps are unique": (
        minute_sample["minute_start"].is_unique
    ),

    # Output minutes must remain chronological.
    "Minutes are chronological": (
        minute_sample["minute_start"].is_monotonic_increasing
    ),

    # A completed bar becomes available one minute after it begins.
    "Availability timing is correct": (
        (
            minute_sample["available_at"]
            - minute_sample["minute_start"]
        )
        == pd.Timedelta(minutes=1)
    ).all(),

    # OHLC values must obey their normal bounds.
    "Midpoint OHLC is valid": (
        (minute_sample["mid_high"] >= minute_sample["mid_open"])
        & (minute_sample["mid_high"] >= minute_sample["mid_close"])
        & (minute_sample["mid_low"] <= minute_sample["mid_open"])
        & (minute_sample["mid_low"] <= minute_sample["mid_close"])
        & (minute_sample["mid_high"] >= minute_sample["mid_low"])
    ).all(),

    # Quoted spreads must remain non-negative.
    "Spread statistics are valid": (
        (minute_sample["spread_open_pips"] >= 0)
        & (minute_sample["spread_mean_pips"] >= 0)
        & (minute_sample["spread_median_pips"] >= 0)
        & (minute_sample["spread_max_pips"] >= 0)
        & (minute_sample["spread_close_pips"] >= 0)
        & (
            minute_sample["spread_max_pips"]
            >= minute_sample["spread_median_pips"]
        )
    ).all(),

    # Imbalance is mathematically bounded.
    "Tick imbalance is bounded": (
        minute_sample["tick_imbalance"]
        .between(-1, 1)
        .all()
    ),

    # A midpoint must equal the average of bid and ask
    # at the opening and closing ticks.
    "Opening midpoint reconciles": np.allclose(
        minute_sample["mid_open"],
        (
            minute_sample["bid_open"]
            + minute_sample["ask_open"]
        ) / 2,
    ),

    "Closing midpoint reconciles": np.allclose(
        minute_sample["mid_close"],
        (
            minute_sample["bid_close"]
            + minute_sample["ask_close"]
        ) / 2,
    ),
}


validation_df = pd.DataFrame(
    {
        "check": validation_checks.keys(),
        "result": [
            "PASS" if result else "FAIL"
            for result in validation_checks.values()
        ],
    }
)


display(validation_df)

print(
    "Overall result:",
    "PASS"
    if all(validation_checks.values())
    else "FAIL",
)

,check,result
0,Tick counts reconcile,PASS
1,Direction counts reconcile,PASS
2,Minute timestamps are unique,PASS
3,Minutes are chronological,PASS
4,Availability timing is correct,PASS
5,Midpoint OHLC is valid,PASS
6,Spread statistics are valid,PASS
7,Tick imbalance is bounded,PASS
8,Opening midpoint reconciles,PASS
9,Closing midpoint reconciles,PASS


Overall result: PASS


In [14]:
# Cell 13 — Prepare the production Parquet output.
#
# This cell does not write any data yet.
# It verifies PyArrow, output paths, and available disk space.

import shutil

import pyarrow as pa
import pyarrow.parquet as pq


MINUTE_OUTPUT_FILE = (
    INTERIM_DATA_DIR
    / "eurusd_1min_2010_2016.parquet"
)

TEMP_MINUTE_OUTPUT_FILE = (
    INTERIM_DATA_DIR
    / "eurusd_1min_2010_2016.partial.parquet"
)


# Make sure the intended output directory exists.
INTERIM_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Check free space on the drive containing the repository.
disk_usage = shutil.disk_usage(PROJECT_ROOT)
free_space_gb = disk_usage.free / (1024 ** 3)


print("PyArrow version:", pa.__version__)
print("Temporary output:", TEMP_MINUTE_OUTPUT_FILE)
print("Final output:    ", MINUTE_OUTPUT_FILE)
print(f"Free disk space:  {free_space_gb:.2f} GB")


# Refuse to overwrite an earlier result automatically.
if MINUTE_OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Final output already exists: {MINUTE_OUTPUT_FILE}"
    )

if TEMP_MINUTE_OUTPUT_FILE.exists():
    raise FileExistsError(
        "A partial output already exists. "
        "Do not delete it until we determine whether it is recoverable: "
        f"{TEMP_MINUTE_OUTPUT_FILE}"
    )

PyArrow version: 25.0.1
Temporary output: C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010_2016.partial.parquet
Final output:     C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010_2016.parquet
Free disk space:  786.21 GB


In [15]:
# Cell 14 — Define the production tick-preparation function.
#
# previous_mid_half_points represents the final midpoint from the
# preceding chunk. Passing it prevents the first tick of each new
# chunk from being incorrectly classified as unchanged.

def prepare_tick_rows(
    raw_ticks,
    previous_mid_half_points=None,
):
    """Add exact timestamps, prices, spreads, and tick directions."""

    prepared = raw_ticks.copy()

    # Convert timestamps and assign minute intervals.
    prepared["event_time"] = pd.to_datetime(
        prepared["timestamp"],
        unit="ms",
        utc=True,
    )

    prepared["minute_start"] = (
        prepared["event_time"].dt.floor("min")
    )

    # Convert quoted prices to exact integer points.
    prepared["ask_points"] = np.rint(
        prepared["askPrice"] * PRICE_SCALE
    ).astype("int64")

    prepared["bid_points"] = np.rint(
        prepared["bidPrice"] * PRICE_SCALE
    ).astype("int64")

    prepared["mid_half_points"] = (
        prepared["ask_points"]
        + prepared["bid_points"]
    )

    # Decimal midpoint and spread values for aggregation.
    prepared["mid"] = (
        prepared["mid_half_points"]
        / (2 * PRICE_SCALE)
    )

    prepared["spread_pips"] = (
        prepared["ask_points"]
        - prepared["bid_points"]
    ) / POINTS_PER_PIP

    midpoint_values = (
        prepared["mid_half_points"].to_numpy()
    )

    # The first row of the complete dataset has no earlier comparison.
    if previous_mid_half_points is None:
        prepend_value = midpoint_values[0]
    else:
        prepend_value = previous_mid_half_points

    midpoint_changes = np.diff(
        midpoint_values,
        prepend=prepend_value,
    )

    prepared["tick_direction"] = np.sign(
        midpoint_changes
    ).astype("int8")

    prepared["is_uptick"] = (
        prepared["tick_direction"] == 1
    )

    prepared["is_downtick"] = (
        prepared["tick_direction"] == -1
    )

    prepared["is_unchanged"] = (
        prepared["tick_direction"] == 0
    )

    prepared["is_locked_quote"] = (
        prepared["ask_points"]
        == prepared["bid_points"]
    )

    return prepared

In [16]:
# Cell 14 verification — this should print PASS.

prepared_test = prepare_tick_rows(
    tick_sample,
    previous_mid_half_points=None,
)

directions_match = np.array_equal(
    prepared_test["tick_direction"].to_numpy(),
    sample_ticks["tick_direction"].to_numpy(),
)

print(
    "Preparation function:",
    "PASS" if directions_match else "FAIL",
)

Preparation function: PASS


In [20]:
# Cell 15 — Corrected reusable one-minute aggregation function.
#
# This version uses the exact column names created in Cell 14
# and preserves the same column order as minute_sample.

def aggregate_minute_rows(prepared_ticks):
    """Aggregate prepared tick rows into one-minute observations."""

    if prepared_ticks.empty:
        return pd.DataFrame()

    minute_bars = (
        prepared_ticks
        .groupby(
            "minute_start",
            sort=True,
            observed=True,
        )
        .agg(
            mid_open=("mid", "first"),
            mid_high=("mid", "max"),
            mid_low=("mid", "min"),
            mid_close=("mid", "last"),

            bid_open=("bidPrice", "first"),
            bid_high=("bidPrice", "max"),
            bid_low=("bidPrice", "min"),
            bid_close=("bidPrice", "last"),

            ask_open=("askPrice", "first"),
            ask_high=("askPrice", "max"),
            ask_low=("askPrice", "min"),
            ask_close=("askPrice", "last"),

            spread_open_pips=("spread_pips", "first"),
            spread_mean_pips=("spread_pips", "mean"),
            spread_median_pips=("spread_pips", "median"),
            spread_max_pips=("spread_pips", "max"),
            spread_close_pips=("spread_pips", "last"),

            tick_count=("timestamp", "size"),
            uptick_count=("is_uptick", "sum"),
            downtick_count=("is_downtick", "sum"),
            unchanged_count=("is_unchanged", "sum"),
            locked_quote_count=("is_locked_quote", "sum"),

            ask_volume_mean=("askVolume", "mean"),
            bid_volume_mean=("bidVolume", "mean"),
            ask_volume_close=("askVolume", "last"),
            bid_volume_close=("bidVolume", "last"),
        )
        .reset_index()
    )

    # A bar covering 12:00:00–12:00:59 becomes available
    # at 12:01:00, after that minute has completed.
    minute_bars["available_at"] = (
        minute_bars["minute_start"]
        + pd.Timedelta(minutes=1)
    )

    # Unchanged ticks are excluded from directional imbalance.
    directional_ticks = (
        minute_bars["uptick_count"]
        + minute_bars["downtick_count"]
    )

    minute_bars["tick_imbalance"] = np.where(
        directional_ticks > 0,
        (
            minute_bars["uptick_count"]
            - minute_bars["downtick_count"]
        ) / directional_ticks,
        0.0,
    )

    minute_bars["ticks_per_second"] = (
        minute_bars["tick_count"] / 60.0
    )

    return minute_bars

In [21]:
# Cell 16 — Verify that the reusable function reproduces
# the previously validated minute bars.

minute_function_test = aggregate_minute_rows(prepared_test)

try:
    pd.testing.assert_frame_equal(
        minute_function_test,
        minute_sample,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )

    print("Aggregation function: PASS")
    print(
        "Minute observations:",
        f"{len(minute_function_test):,}",
    )

except AssertionError as error:
    print("Aggregation function: FAIL")
    print()
    print(error)

Aggregation function: PASS
Minute observations: 5,008


In [22]:
# Cell 17 — Test Parquet serialization in memory.
#
# No file is written by this cell.
# It confirms that PyArrow can serialize and recover our
# complete minute-bar structure before the production run.

sample_arrow_table = pa.Table.from_pandas(
    minute_function_test,
    preserve_index=False,
)

memory_output = pa.BufferOutputStream()

pq.write_table(
    sample_arrow_table,
    memory_output,
    compression="zstd",
)

parquet_buffer = memory_output.getvalue()

# Read the compressed data back from memory.
recovered_table = pq.read_table(
    pa.BufferReader(parquet_buffer)
)

schema_matches = (
    recovered_table.schema
    == sample_arrow_table.schema
)

row_count_matches = (
    recovered_table.num_rows
    == len(minute_function_test)
)


print("Rows serialized:", f"{recovered_table.num_rows:,}")
print(
    "Compressed sample size:",
    f"{len(parquet_buffer) / (1024 ** 2):.2f} MB",
)
print("Schema preserved:", schema_matches)
print("Row count preserved:", row_count_matches)

print(
    "\nParquet serialization:",
    "PASS"
    if schema_matches and row_count_matches
    else "FAIL",
)

Rows serialized: 5,008
Compressed sample size: 0.36 MB
Schema preserved: True
Row count preserved: True

Parquet serialization: PASS


In [23]:
# Cell 18 — Create a chunk-safe tick-to-minute processor.
#
# A CSV chunk can end halfway through a minute.
# We therefore hold back its final minute and combine it with
# the beginning of the next chunk.
#
# The function yields only completed groups of minute bars.
# At the end of the input, it also yields the final carried minute.

RAW_TICK_COLUMNS = list(TICK_DTYPES.keys())


def produce_minute_batches(raw_chunks):
    """Convert an iterable of raw tick chunks into minute-bar batches."""

    carry_raw = None

    # Midpoint immediately before the first carried tick.
    # This preserves tick direction across chunk boundaries.
    carry_previous_mid = None

    for raw_chunk in raw_chunks:

        raw_chunk = (
            raw_chunk[RAW_TICK_COLUMNS]
            .reset_index(drop=True)
        )

        # Reattach the unfinished minute from the prior chunk.
        if carry_raw is not None:
            combined_raw = pd.concat(
                [carry_raw, raw_chunk],
                ignore_index=True,
            )
        else:
            combined_raw = raw_chunk

        prepared = prepare_tick_rows(
            combined_raw,
            previous_mid_half_points=carry_previous_mid,
        )

        # The final minute may continue in the next CSV chunk,
        # so it must not be aggregated yet.
        final_minute = prepared["minute_start"].iloc[-1]

        carry_mask = (
            prepared["minute_start"] == final_minute
        )

        first_carry_position = int(
            np.flatnonzero(carry_mask.to_numpy())[0]
        )

        completed_ticks = prepared.iloc[
            :first_carry_position
        ]

        # Save the midpoint immediately preceding the new carry.
        if first_carry_position > 0:
            next_carry_previous_mid = int(
                prepared["mid_half_points"].iloc[
                    first_carry_position - 1
                ]
            )
        else:
            next_carry_previous_mid = carry_previous_mid

        # Keep only the original raw columns for the next cycle.
        carry_raw = (
            combined_raw
            .iloc[first_carry_position:]
            [RAW_TICK_COLUMNS]
            .reset_index(drop=True)
        )

        carry_previous_mid = next_carry_previous_mid

        if not completed_ticks.empty:
            yield aggregate_minute_rows(completed_ticks)

    # The file has ended, so the remaining minute can now
    # be treated as the terminal bar.
    if carry_raw is not None and not carry_raw.empty:
        final_prepared = prepare_tick_rows(
            carry_raw,
            previous_mid_half_points=carry_previous_mid,
        )

        yield aggregate_minute_rows(final_prepared)

In [24]:
# Cell 19 — Test the chunk-carry logic.
#
# The chunked result must exactly reproduce the result obtained
# by processing all 100,000 sample ticks at once.

artificial_chunks = [
    tick_sample.iloc[:12_345].copy(),
    tick_sample.iloc[12_345:54_321].copy(),
    tick_sample.iloc[54_321:87_654].copy(),
    tick_sample.iloc[87_654:].copy(),
]

chunked_minute_batches = list(
    produce_minute_batches(artificial_chunks)
)

chunked_minute_test = pd.concat(
    chunked_minute_batches,
    ignore_index=True,
)


try:
    pd.testing.assert_frame_equal(
        chunked_minute_test,
        minute_function_test,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )

    print("Chunk-boundary test: PASS")
    print(
        "Input chunks:",
        len(artificial_chunks),
    )
    print(
        "Minute observations:",
        f"{len(chunked_minute_test):,}",
    )

except AssertionError as error:
    print("Chunk-boundary test: FAIL")
    print()
    print(error)

Chunk-boundary test: PASS
Input chunks: 4
Minute observations: 5,008


In [25]:
# Cell 20 — Final production-run preflight.
#
# This cell still does not process or write the full dataset.
# It locks the tested schema and checks that both output paths
# are clear.

CSV_CHUNK_SIZE = 1_000_000

PARQUET_SCHEMA = pa.Table.from_pandas(
    minute_function_test,
    preserve_index=False,
).schema


# Recheck immediately before the production run.
if MINUTE_OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Final output already exists: {MINUTE_OUTPUT_FILE}"
    )

if TEMP_MINUTE_OUTPUT_FILE.exists():
    raise FileExistsError(
        "A partial production file already exists: "
        f"{TEMP_MINUTE_OUTPUT_FILE}"
    )


estimated_chunks = int(
    np.ceil(123_790_754 / CSV_CHUNK_SIZE)
)


print("Production preflight: PASS")
print("CSV chunk size:      ", f"{CSV_CHUNK_SIZE:,}")
print("Estimated chunks:    ", estimated_chunks)
print("Parquet columns:      ", len(PARQUET_SCHEMA))
print("Compression:          ", "Zstandard")
print("Temporary output:     ", TEMP_MINUTE_OUTPUT_FILE)
print("Final output:         ", MINUTE_OUTPUT_FILE)

Production preflight: PASS
CSV chunk size:       1,000,000
Estimated chunks:     124
Parquet columns:       30
Compression:           Zstandard
Temporary output:      C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010_2016.partial.parquet
Final output:          C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010_2016.parquet


In [26]:
# Cell 21 — Build the production one-minute Parquet dataset.
#
# This reads the 4.4 GB CSV in 1,000,000-row chunks.
# It writes only to the temporary .partial.parquet file.
#
# The temporary file is renamed to the final filename only if:
#   1. every chunk finishes,
#   2. all batches remain chronological,
#   3. every audited tick is represented.

import time


EXPECTED_TICK_ROWS = 123_790_754

writer = None
output_batch_count = 0
ticks_written = 0
minutes_written = 0
last_minute_written = None

start_time = time.perf_counter()


try:
    raw_chunk_reader = pd.read_csv(
        MAIN_RAW_FILE,
        dtype=TICK_DTYPES,
        chunksize=CSV_CHUNK_SIZE,
    )

    writer = pq.ParquetWriter(
        TEMP_MINUTE_OUTPUT_FILE,
        PARQUET_SCHEMA,
        compression="zstd",
    )

    for minute_batch in produce_minute_batches(
        raw_chunk_reader
    ):
        output_batch_count += 1

        # Confirm that each individual batch is ordered and unique.
        if not minute_batch["minute_start"].is_monotonic_increasing:
            raise ValueError(
                "Minute timestamps are not ordered within a batch."
            )

        if minute_batch["minute_start"].duplicated().any():
            raise ValueError(
                "Duplicate minute timestamps found within a batch."
            )

        # Confirm that this batch starts after the preceding batch.
        current_first_minute = (
            minute_batch["minute_start"].iloc[0]
        )

        if (
            last_minute_written is not None
            and current_first_minute <= last_minute_written
        ):
            raise ValueError(
                "Minute batches overlap or are out of order."
            )

        # Confirm the availability timestamp convention.
        expected_availability = (
            minute_batch["minute_start"]
            + pd.Timedelta(minutes=1)
        )

        if not minute_batch["available_at"].equals(
            expected_availability
        ):
            raise ValueError(
                "An invalid available_at timestamp was found."
            )

        # Enforce the schema tested in Cell 17.
        arrow_batch = pa.Table.from_pandas(
            minute_batch,
            schema=PARQUET_SCHEMA,
            preserve_index=False,
            safe=True,
        )

        writer.write_table(arrow_batch)

        ticks_written += int(
            minute_batch["tick_count"].sum()
        )
        minutes_written += len(minute_batch)

        last_minute_written = (
            minute_batch["minute_start"].iloc[-1]
        )

        # Print occasional progress without flooding the notebook.
        if output_batch_count % 10 == 0:
            elapsed = time.perf_counter() - start_time
            tick_rate = ticks_written / elapsed

            print(
                f"Output batch {output_batch_count:,} | "
                f"ticks {ticks_written:,} | "
                f"minutes {minutes_written:,} | "
                f"{tick_rate:,.0f} ticks/sec"
            )

    # Finish and close the Parquet footer before validation.
    writer.close()
    writer = None

    # The minute tick counts must reconcile exactly with the
    # full-file audit from Cell 8.
    if ticks_written != EXPECTED_TICK_ROWS:
        raise ValueError(
            "Tick reconciliation failed: "
            f"expected {EXPECTED_TICK_ROWS:,}, "
            f"wrote {ticks_written:,}."
        )

    # Refuse to replace a final file that appeared during the run.
    if MINUTE_OUTPUT_FILE.exists():
        raise FileExistsError(
            f"Final output now exists: {MINUTE_OUTPUT_FILE}"
        )

    # Promote the completed temporary file to its final name.
    TEMP_MINUTE_OUTPUT_FILE.rename(
        MINUTE_OUTPUT_FILE
    )

except Exception:
    # Closing the writer preserves a readable partial file for
    # diagnosis if the run fails.
    if writer is not None:
        writer.close()

    print()
    print("Production conversion: FAILED")
    print(
        "Partial output retained at:",
        TEMP_MINUTE_OUTPUT_FILE,
    )

    raise


elapsed = time.perf_counter() - start_time
output_size_mb = (
    MINUTE_OUTPUT_FILE.stat().st_size
    / (1024 ** 2)
)


print()
print("Production conversion: PASS")
print("Ticks reconciled:     ", f"{ticks_written:,}")
print("Minutes written:      ", f"{minutes_written:,}")
print("First minute:         ", minute_function_test["minute_start"].iloc[0])
print("Last minute:          ", last_minute_written)
print("Output size:          ", f"{output_size_mb:,.2f} MB")
print("Elapsed time:         ", f"{elapsed / 60:.2f} minutes")
print("Final output:         ", MINUTE_OUTPUT_FILE)

Output batch 10 | ticks 9,999,993 | minutes 400,592 | 1,829,061 ticks/sec
Output batch 20 | ticks 19,999,996 | minutes 558,908 | 1,895,019 ticks/sec
Output batch 30 | ticks 29,999,969 | minutes 680,772 | 1,933,363 ticks/sec
Output batch 40 | ticks 39,999,976 | minutes 813,029 | 1,955,120 ticks/sec
Output batch 50 | ticks 49,999,934 | minutes 959,954 | 1,971,552 ticks/sec
Output batch 60 | ticks 59,999,957 | minutes 1,160,044 | 1,971,897 ticks/sec
Output batch 70 | ticks 69,999,961 | minutes 1,349,299 | 1,977,956 ticks/sec
Output batch 80 | ticks 79,999,984 | minutes 1,595,350 | 1,969,223 ticks/sec
Output batch 90 | ticks 89,999,959 | minutes 1,795,294 | 1,972,701 ticks/sec
Output batch 100 | ticks 99,999,873 | minutes 1,953,487 | 1,969,206 ticks/sec
Output batch 110 | ticks 109,999,937 | minutes 2,103,133 | 1,978,014 ticks/sec
Output batch 120 | ticks 119,999,848 | minutes 2,248,971 | 1,979,759 ticks/sec

Production conversion: PASS
Ticks reconciled:      123,790,754
Minutes written:  

In [27]:
# Cell 22 — Independently inspect the completed Parquet file.
#
# This reads only Parquet metadata, not all 2.3 million rows.
# It confirms that the file was closed correctly and that its
# footer and schema are readable.

parquet_file = pq.ParquetFile(
    MINUTE_OUTPUT_FILE
)

parquet_metadata = parquet_file.metadata


print("Parquet metadata check")
print("----------------------")
print("Rows:           ", f"{parquet_metadata.num_rows:,}")
print("Columns:        ", parquet_metadata.num_columns)
print("Row groups:     ", parquet_metadata.num_row_groups)
print("Created by:     ", parquet_metadata.created_by)
print("Final exists:   ", MINUTE_OUTPUT_FILE.exists())
print("Partial exists: ", TEMP_MINUTE_OUTPUT_FILE.exists())


metadata_pass = (
    parquet_metadata.num_rows == minutes_written
    and parquet_metadata.num_columns == len(PARQUET_SCHEMA)
    and MINUTE_OUTPUT_FILE.exists()
    and not TEMP_MINUTE_OUTPUT_FILE.exists()
)


print()
print(
    "Metadata validation:",
    "PASS" if metadata_pass else "FAIL",
)

Parquet metadata check
----------------------
Rows:            2,295,731
Columns:         30
Row groups:      125
Created by:      parquet-cpp-arrow version 25.0.1
Final exists:    True
Partial exists:  False

Metadata validation: PASS


In [28]:
# Cell 23 — Validate core production columns from the saved file.
#
# Only three columns are loaded, keeping memory use modest.
# This independently checks:
#   - row count,
#   - chronological order,
#   - duplicate minutes,
#   - bar availability timing,
#   - positive tick counts,
#   - exact reconciliation to the raw tick total.

saved_core = pd.read_parquet(
    MINUTE_OUTPUT_FILE,
    columns=[
        "minute_start",
        "available_at",
        "tick_count",
    ],
)


saved_tick_total = int(
    saved_core["tick_count"].sum()
)

row_count_ok = (
    len(saved_core) == parquet_metadata.num_rows
)

chronology_ok = (
    saved_core["minute_start"].is_monotonic_increasing
)

unique_minutes_ok = (
    not saved_core["minute_start"].duplicated().any()
)

availability_ok = (
    saved_core["available_at"]
    == saved_core["minute_start"]
    + pd.Timedelta(minutes=1)
).all()

positive_tick_counts_ok = (
    saved_core["tick_count"] > 0
).all()

tick_reconciliation_ok = (
    saved_tick_total == EXPECTED_TICK_ROWS
)

boundaries_ok = (
    saved_core["minute_start"].iloc[0]
    == minute_function_test["minute_start"].iloc[0]
    and saved_core["minute_start"].iloc[-1]
    == last_minute_written
)


validation_results = {
    "Row count": row_count_ok,
    "Chronological order": chronology_ok,
    "Unique minutes": unique_minutes_ok,
    "Availability timing": availability_ok,
    "Positive tick counts": positive_tick_counts_ok,
    "Tick reconciliation": tick_reconciliation_ok,
    "Timestamp boundaries": boundaries_ok,
}


for check_name, passed in validation_results.items():
    print(
        f"{check_name:<24}",
        "PASS" if passed else "FAIL",
    )


print()
print("Saved rows:       ", f"{len(saved_core):,}")
print("Saved tick total: ", f"{saved_tick_total:,}")
print(
    "First minute:     ",
    saved_core["minute_start"].iloc[0],
)
print(
    "Last minute:      ",
    saved_core["minute_start"].iloc[-1],
)

print()
print(
    "Core file validation:",
    (
        "PASS"
        if all(validation_results.values())
        else "FAIL"
    ),
)

Row count                PASS
Chronological order      PASS
Unique minutes           PASS
Availability timing      PASS
Positive tick counts     PASS
Tick reconciliation      PASS
Timestamp boundaries     PASS

Saved rows:        2,295,731
Saved tick total:  123,790,754
First minute:      2010-01-01 00:00:00+00:00
Last minute:       2016-04-05 05:59:00+00:00

Core file validation: PASS


In [29]:
# Cell 24 — Identify gaps between observed trading minutes.
#
# A gap exists whenever two consecutive observations are more
# than one minute apart.
#
# This cell only measures gaps. It does not assume that missing
# weekends, holidays, or inactive periods are data errors.

minute_spacing = (
    saved_core["minute_start"].diff()
)

gap_positions = np.flatnonzero(
    minute_spacing > pd.Timedelta(minutes=1)
)


gap_records = []

for position in gap_positions:
    previous_minute = saved_core[
        "minute_start"
    ].iloc[position - 1]

    next_minute = saved_core[
        "minute_start"
    ].iloc[position]

    elapsed_minutes = int(
        (
            next_minute - previous_minute
        ).total_seconds() // 60
    )

    gap_records.append(
        {
            "previous_observed": previous_minute,
            "next_observed": next_minute,
            "first_missing": (
                previous_minute
                + pd.Timedelta(minutes=1)
            ),
            "last_missing": (
                next_minute
                - pd.Timedelta(minutes=1)
            ),
            "missing_minutes": elapsed_minutes - 1,
        }
    )


gap_table = pd.DataFrame(gap_records)

total_missing_minutes = int(
    gap_table["missing_minutes"].sum()
)

short_gap_count = int(
    (gap_table["missing_minutes"] <= 5).sum()
)

medium_gap_count = int(
    (
        (gap_table["missing_minutes"] > 5)
        & (gap_table["missing_minutes"] <= 60)
    ).sum()
)

long_gap_count = int(
    (gap_table["missing_minutes"] > 60).sum()
)


print("Gap summary")
print("-----------")
print("Number of gaps:        ", f"{len(gap_table):,}")
print("Total missing minutes: ", f"{total_missing_minutes:,}")
print("Short gaps, 1–5 min:   ", f"{short_gap_count:,}")
print("Medium gaps, 6–60 min: ", f"{medium_gap_count:,}")
print("Long gaps, over 60 min:", f"{long_gap_count:,}")

print()
print("Largest 15 gaps:")

display(
    gap_table
    .sort_values(
        "missing_minutes",
        ascending=False,
    )
    .head(15)
    .reset_index(drop=True)
)

Gap summary
-----------
Number of gaps:         3,933
Total missing minutes:  996,469
Short gaps, 1–5 min:    2,922
Medium gaps, 6–60 min:  644
Long gaps, over 60 min: 367

Largest 15 gaps:


,previous_observed,next_observed,first_missing,last_missing,missing_minutes
0,2015-12-31 21:59:00+00:00,2016-01-03 22:00:00+00:00,2015-12-31 22:00:00+00:00,2016-01-03 21:59:00+00:00,4320
1,2015-12-25 07:59:00+00:00,2015-12-27 22:02:00+00:00,2015-12-25 08:00:00+00:00,2015-12-27 22:01:00+00:00,3722
2,2010-02-26 21:59:00+00:00,2010-03-01 00:00:00+00:00,2010-02-26 22:00:00+00:00,2010-02-28 23:59:00+00:00,3000
3,2011-11-04 20:59:00+00:00,2011-11-06 22:00:00+00:00,2011-11-04 21:00:00+00:00,2011-11-06 21:59:00+00:00,2940
4,2014-07-25 20:59:00+00:00,2014-07-27 22:00:00+00:00,2014-07-25 21:00:00+00:00,2014-07-27 21:59:00+00:00,2940
5,2013-11-15 21:59:00+00:00,2013-11-17 23:00:00+00:00,2013-11-15 22:00:00+00:00,2013-11-17 22:59:00+00:00,2940
6,2010-11-05 20:59:00+00:00,2010-11-07 22:00:00+00:00,2010-11-05 21:00:00+00:00,2010-11-07 21:59:00+00:00,2940
7,2010-06-04 20:59:00+00:00,2010-06-06 22:00:00+00:00,2010-06-04 21:00:00+00:00,2010-06-06 21:59:00+00:00,2940
8,2010-09-10 20:59:00+00:00,2010-09-12 22:00:00+00:00,2010-09-10 21:00:00+00:00,2010-09-12 21:59:00+00:00,2940
9,2014-10-31 20:59:00+00:00,2014-11-02 22:00:00+00:00,2014-10-31 21:00:00+00:00,2014-11-02 21:59:00+00:00,2940


In [30]:
# Cell 25 — Measure where the missing time comes from.
#
# A large number of small gaps can sound concerning, but long
# weekend and holiday closures usually account for nearly all
# missing calendar time.
#
# We separate:
#   - isolated quiet minutes,
#   - short interruptions,
#   - longer closures,
#   - weekend-sized closures.

gap_bins = [
    0,
    1,
    5,
    15,
    60,
    180,
    720,
    1_440,
    np.inf,
]

gap_labels = [
    "1 minute",
    "2–5 minutes",
    "6–15 minutes",
    "16–60 minutes",
    "61–180 minutes",
    "181–720 minutes",
    "721–1,440 minutes",
    "Over 1,440 minutes",
]


gap_table["duration_group"] = pd.cut(
    gap_table["missing_minutes"],
    bins=gap_bins,
    labels=gap_labels,
    include_lowest=True,
)


gap_size_summary = (
    gap_table
    .groupby(
        "duration_group",
        observed=False,
    )
    .agg(
        gap_count=("missing_minutes", "size"),
        missing_minutes=("missing_minutes", "sum"),
    )
    .reset_index()
)


gap_size_summary["share_of_missing_time"] = (
    100
    * gap_size_summary["missing_minutes"]
    / gap_size_summary["missing_minutes"].sum()
)


display(gap_size_summary)


long_gap_missing_minutes = int(
    gap_table.loc[
        gap_table["missing_minutes"] > 1_440,
        "missing_minutes",
    ].sum()
)

long_gap_share = (
    100
    * long_gap_missing_minutes
    / total_missing_minutes
)


print(
    "Share of missing time caused by gaps over one day:",
    f"{long_gap_share:.2f}%",
)

,duration_group,gap_count,missing_minutes,share_of_missing_time
0,1 minute,2547,2547,0.255603
1,2–5 minutes,375,906,0.090921
2,6–15 minutes,14,124,0.012444
3,16–60 minutes,630,37602,3.773524
4,61–180 minutes,32,3707,0.372014
5,181–720 minutes,3,831,0.083394
6,"721–1,440 minutes",5,5960,0.598112
7,"Over 1,440 minutes",327,944792,94.813988


Share of missing time caused by gaps over one day: 94.81%


In [31]:
# Cell 26 — Inspect the timing of 16–60 minute gaps.
#
# If these gaps repeatedly occur at the same UTC time, that
# supports the interpretation that they are scheduled rollover
# or infrastructure-maintenance periods rather than random
# corruption.

rollover_gap_candidates = gap_table.loc[
    gap_table["missing_minutes"].between(
        16,
        60,
    )
].copy()


rollover_gap_candidates["previous_weekday"] = (
    rollover_gap_candidates["previous_observed"]
    .dt.day_name()
)

rollover_gap_candidates["previous_time_utc"] = (
    rollover_gap_candidates["previous_observed"]
    .dt.strftime("%H:%M")
)

rollover_gap_candidates["next_time_utc"] = (
    rollover_gap_candidates["next_observed"]
    .dt.strftime("%H:%M")
)


rollover_pattern_summary = (
    rollover_gap_candidates
    .groupby(
        [
            "missing_minutes",
            "previous_time_utc",
            "next_time_utc",
        ],
        observed=True,
    )
    .size()
    .rename("occurrences")
    .reset_index()
    .sort_values(
        "occurrences",
        ascending=False,
    )
)


print(
    "Gaps inspected:",
    f"{len(rollover_gap_candidates):,}",
)

print()
print("Most common UTC timing patterns:")

display(
    rollover_pattern_summary
    .head(20)
    .reset_index(drop=True)
)

Gaps inspected: 630

Most common UTC timing patterns:


,missing_minutes,previous_time_utc,next_time_utc,occurrences
0,60,01:59,03:00,35
1,60,10:59,12:00,34
2,60,12:59,14:00,32
3,60,00:59,02:00,31
4,60,06:59,08:00,31
5,60,18:59,20:00,30
6,60,14:59,16:00,29
7,60,07:59,09:00,29
8,60,02:59,04:00,28
9,60,05:59,07:00,27


In [32]:
# Cell 27 — Profile exact one-hour data gaps.
#
# A gap of 60 missing minutes between HH:59 and HH+2:00
# represents one absent clock-hour block.
#
# EUR/USD should not normally have zero ticks for a full hour
# during an active weekday session, so these are candidates for
# later re-download and repair.

one_hour_gaps = gap_table.loc[
    gap_table["missing_minutes"] == 60
].copy()


one_hour_gaps["missing_hour_start"] = (
    one_hour_gaps["first_missing"]
)

one_hour_gaps["year"] = (
    one_hour_gaps["missing_hour_start"].dt.year
)

one_hour_gaps["weekday"] = (
    one_hour_gaps["missing_hour_start"].dt.day_name()
)

one_hour_gaps["hour_utc"] = (
    one_hour_gaps["missing_hour_start"].dt.hour
)


exact_hour_boundary = (
    one_hour_gaps["missing_hour_start"].dt.minute.eq(0)
    & one_hour_gaps["last_missing"].dt.minute.eq(59)
)


print("Exact one-hour gaps:       ", f"{len(one_hour_gaps):,}")
print(
    "Start and end on full-hour boundaries:",
    f"{exact_hour_boundary.mean():.2%}",
)

print()
print("One-hour gaps by year:")

display(
    one_hour_gaps
    .groupby("year")
    .size()
    .rename("gap_count")
    .reset_index()
)

print()
print("One-hour gaps by weekday:")

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

display(
    one_hour_gaps["weekday"]
    .value_counts()
    .reindex(weekday_order, fill_value=0)
    .rename_axis("weekday")
    .reset_index(name="gap_count")
)

Exact one-hour gaps:        620
Start and end on full-hour boundaries: 100.00%

One-hour gaps by year:


,year,gap_count
0,2010,262
1,2011,26
2,2012,73
3,2013,83
4,2014,106
5,2015,60
6,2016,10



One-hour gaps by weekday:


,weekday,gap_count
0,Monday,125
1,Tuesday,129
2,Wednesday,121
3,Thursday,119
4,Friday,117
5,Saturday,0
6,Sunday,9


In [33]:
# Cell 28 — Save a repair manifest for missing hourly blocks.
#
# This small CSV lists the exact Dukascopy hours that should be
# requested again later.
#
# It does not alter the raw data or the one-minute Parquet file.

REPORT_TABLES_DIR = (
    PROJECT_ROOT
    / "reports"
    / "tables"
)

REPORT_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MISSING_HOUR_MANIFEST_FILE = (
    REPORT_TABLES_DIR
    / "eurusd_missing_hour_manifest_2010_2016.csv"
)


missing_hour_manifest = (
    one_hour_gaps[
        [
            "first_missing",
            "last_missing",
            "missing_minutes",
            "year",
            "weekday",
            "hour_utc",
        ]
    ]
    .rename(
        columns={
            "first_missing": "missing_hour_start_utc",
            "last_missing": "missing_hour_end_utc",
        }
    )
    .sort_values("missing_hour_start_utc")
    .reset_index(drop=True)
)


# Add context that will make the file understandable by itself.
missing_hour_manifest.insert(
    0,
    "instrument",
    "EURUSD",
)

missing_hour_manifest["status"] = (
    "candidate_missing_source_block"
)


if MISSING_HOUR_MANIFEST_FILE.exists():
    raise FileExistsError(
        "Repair manifest already exists: "
        f"{MISSING_HOUR_MANIFEST_FILE}"
    )


missing_hour_manifest.to_csv(
    MISSING_HOUR_MANIFEST_FILE,
    index=False,
)


print("Repair manifest created: PASS")
print(
    "Missing hourly blocks:",
    f"{len(missing_hour_manifest):,}",
)
print(
    "Manifest size:",
    f"{MISSING_HOUR_MANIFEST_FILE.stat().st_size / 1024:.2f} KB",
)
print("Saved to:", MISSING_HOUR_MANIFEST_FILE)

Repair manifest created: PASS
Missing hourly blocks: 620
Manifest size: 66.54 KB
Saved to: C:\Users\Ijzd4\trend models\EURUSD 15\reports\tables\eurusd_missing_hour_manifest_2010_2016.csv


In [34]:
# Cell 29 — Record the final audit conclusion.
#
# PASS WITH COVERAGE WARNINGS means:
#   - all available tick rows passed structural validation,
#   - minute aggregation and Parquet output passed,
#   - known timestamp gaps must be respected,
#   - missing hourly blocks can be repaired later.

AUDIT_STATUS = "PASS WITH COVERAGE WARNINGS"


print("EUR/USD tick-data audit")
print("========================")
print("Status:", AUDIT_STATUS)

print()
print("Available raw ticks:       ", f"{EXPECTED_TICK_ROWS:,}")
print("One-minute observations:   ", f"{len(saved_core):,}")
print(
    "Observed coverage:         ",
    saved_core["minute_start"].iloc[0],
    "to",
    saved_core["minute_start"].iloc[-1],
)
print("Exact missing-hour blocks: ", f"{len(missing_hour_manifest):,}")
print("Locked quotes retained:    ", "770")
print("Maximum spread retained:   ", "34.30 pips")

print()
print("Model-development rules:")
print("1. Do not create artificial bars for missing minutes.")
print("2. Do not forward-fill prices across timestamp gaps.")
print("3. Require exact timestamps for lagged features.")
print("4. Require an exact t + 15 minute price for each target.")
print("5. Exclude samples whose required window crosses a gap.")
print("6. Preserve spread spikes as potential regime information.")

print()
print("Production minute dataset:")
print(MINUTE_OUTPUT_FILE)

print()
print("Hourly repair manifest:")
print(MISSING_HOUR_MANIFEST_FILE)

EUR/USD tick-data audit
Status: PASS WITH COVERAGE WARNINGS

Available raw ticks:        123,790,754
One-minute observations:    2,295,731
Observed coverage:          2010-01-01 00:00:00+00:00 to 2016-04-05 05:59:00+00:00
Exact missing-hour blocks:  620
Locked quotes retained:     770
Maximum spread retained:    34.30 pips

Model-development rules:
1. Do not create artificial bars for missing minutes.
2. Do not forward-fill prices across timestamp gaps.
3. Require exact timestamps for lagged features.
4. Require an exact t + 15 minute price for each target.
5. Exclude samples whose required window crosses a gap.
6. Preserve spread spikes as potential regime information.

Production minute dataset:
C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010_2016.parquet

Hourly repair manifest:
C:\Users\Ijzd4\trend models\EURUSD 15\reports\tables\eurusd_missing_hour_manifest_2010_2016.csv
